# Passing Quality — 3 Model Comparison

**Naive**: `from_PQ` only

**Direct**: `from_PQ` + 7 delta TQ (model sees pre-quality directly)

**Indirect**: 7 delta TQ only (model never sees `from_PQ` — does tactical shift alone predict the change?)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_DIR = "../../../../thesis_data/processed_data/thesis_model_dataset/active/"
df = pd.read_parquet(DATA_DIR + "within_league_transfers_v5.parquet")
mf = df[df["from_position"] == "Midfielder"].copy()

mf["delta_PQ"] = mf["to_Passing quality"] - mf["from_Passing quality"]

TEAM_QUALITIES = ["ATTACK", "ATTACKING_TRANSITION", "CHANCE_CREATION",
                  "DEFENCE", "DEFENSIVE_TRANSITION", "OUTCOME", "PENETRATION"]

for tq in TEAM_QUALITIES:
    mf[f"delta_tq_{tq}"] = mf[f"to_q_{tq}"] - mf[f"from_q_proj_{tq}"]

delta_tq = [f"delta_tq_{tq}" for tq in TEAM_QUALITIES]
mf = mf.dropna(subset=["from_Passing quality", "delta_PQ"] + delta_tq)

train, test = train_test_split(mf, test_size=0.2, random_state=42)
y_train = train["delta_PQ"]
y_test  = test["delta_PQ"]
print(f"Train: {len(train):,}  |  Test: {len(test):,}")

---
## Model 1: Naive

In [ ]:
X_tr = sm.add_constant(train[["from_Passing quality"]])
X_te = sm.add_constant(test[["from_Passing quality"]])

naive = sm.OLS(y_train, X_tr).fit()
naive_pred = naive.predict(X_te)

print(naive.summary())

---
## Model 2: Direct (from_PQ + all delta TQ)

In [ ]:
feat_direct = ["from_Passing quality"] + delta_tq
X_tr = sm.add_constant(train[feat_direct])
X_te = sm.add_constant(test[feat_direct])

direct = sm.OLS(y_train, X_tr).fit()
direct_pred = direct.predict(X_te)

print(direct.summary())

---
## Model 3: Indirect (delta TQ only, no from_PQ)

In [ ]:
X_tr = sm.add_constant(train[delta_tq])
X_te = sm.add_constant(test[delta_tq])

indirect = sm.OLS(y_train, X_tr).fit()
indirect_pred = indirect.predict(X_te)

print(indirect.summary())

---
## Comparison

In [ ]:
models = {
    "Naive (from_PQ)": (naive, naive_pred),
    "Direct (from_PQ + 7 delta TQ)": (direct, direct_pred),
    "Indirect (7 delta TQ only)": (indirect, indirect_pred),
}

rows = []
for name, (model, pred) in models.items():
    rows.append({
        "Model": name,
        "R2_test": r2_score(y_test, pred),
        "MAE_test": mean_absolute_error(y_test, pred),
        "R2_adj_train": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic,
    })

comp = pd.DataFrame(rows)
print(comp.to_string(index=False, float_format="{:.4f}".format))

---
## Coefficients: Direct vs Indirect

In [ ]:
coef_direct = pd.DataFrame({
    "Feature": direct.params.index,
    "Coef_direct": direct.params.values,
    "p_direct": direct.pvalues.values,
})

coef_indirect = pd.DataFrame({
    "Feature": indirect.params.index,
    "Coef_indirect": indirect.params.values,
    "p_indirect": indirect.pvalues.values,
})

coef_comp = coef_direct.merge(coef_indirect, on='Feature', how='outer').fillna('-')
print(coef_comp.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, (_, pred)) in zip(axes, models.items()):
    ax.scatter(y_test, pred, alpha=0.15, s=10)
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, "r--", linewidth=0.5)
    ax.set_xlabel("Actual delta")
    ax.set_ylabel("Predicted delta")
    ax.set_title(name)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

---
## Case Study: Declan Rice (West Ham → Arsenal)

In [ ]:
rice = mf[mf['wy_player_id'] == 379209].iloc[0]
pre  = rice["from_Passing quality"]
post = rice["to_Passing quality"]
delta_actual = post - pre

print("=" * 60)
print("  TEAM TACTICAL QUALITIES")
print("=" * 60)
header = "  {:<22} {:>10} {:>10} {:>10}".format("Quality", "West Ham", "Arsenal", "Delta")
print(header)
print("-" * 60)
for tq in TEAM_QUALITIES:
    fr = rice[f"from_q_proj_{tq}"]
    to = rice[f"to_q_{tq}"]
    print("  {:<22} {:>10.3f} {:>10.3f} {:>+10.3f}".format(tq, fr, to, to - fr))
print()

# Predictions
rice_naive = sm.add_constant(pd.DataFrame({"from_Passing quality": [pre]}))
rice_direct = sm.add_constant(pd.DataFrame(
    {"from_Passing quality": [pre], **{f"delta_tq_{tq}": [rice[f"delta_tq_{tq}"]] for tq in TEAM_QUALITIES}}
))
rice_indirect = sm.add_constant(pd.DataFrame(
    {f"delta_tq_{tq}": [rice[f"delta_tq_{tq}"]] for tq in TEAM_QUALITIES}
))

pn = naive.predict(rice_naive)[0]
pd_ = direct.predict(rice_direct)[0]
pi = indirect.predict(rice_indirect)[0]

print("=" * 60)
print("  PREDICTIONS")
print("=" * 60)
print("  {:<35} {:>8} {:>10}".format("Model", "Delta", "to_PQ"))
print("-" * 60)
print("  {:<35} {:>+8.3f} {:>10.3f}".format("Naive", pn, pre + pn))
print("  {:<35} {:>+8.3f} {:>10.3f}".format("Direct (from_PQ + delta TQ)", pd_, pre + pd_))
print("  {:<35} {:>+8.3f} {:>10.3f}".format("Indirect (delta TQ only)", pi, pre + pi))
print("  {:<35} {:>+8.3f} {:>10.3f}".format("Actual", delta_actual, post))